In [1]:
import pandas as pd 
import numpy as np

import joblib

pd.set_option("display.max_columns", None)

In [3]:
df = pd.read_csv("../processed_data/forecasting_features.csv")

df["date"] = pd.to_datetime(df["date"])

df.head()

,date,store,item,sales,year,month,quarter,day,day_of_week,day_of_year,week_of_year,is_weekend,lag_7,lag_14,lag_30,lag_90,rolling_mean_7,rolling_mean_30,rolling_std_7,rolling_std_30
0,2013-04-01,1,1,11,2013,4,2,1,0,91,14,0,13.0,11.0,13.0,13.0,15.571429,15.400000,3.359422,3.596934
1,2013-04-02,1,1,19,2013,4,2,2,1,92,14,0,16.0,19.0,20.0,11.0,15.285714,15.333333,3.683942,3.660915
2,2013-04-03,1,1,24,2013,4,2,3,2,93,14,0,11.0,14.0,14.0,14.0,15.714286,15.300000,3.946065,3.621297
3,2013-04-04,1,1,18,2013,4,2,4,3,94,14,0,13.0,17.0,13.0,13.0,17.571429,15.633333,4.391550,3.943422
4,2013-04-05,1,1,19,2013,4,2,5,4,95,14,0,17.0,21.0,17.0,10.0,18.285714,15.800000,3.903600,3.933937


In [6]:
model = joblib.load(
    "../models/xgboost_forecasting_model.pkl"
)

type(model)

xgboost.sklearn.XGBRegressor

In [7]:
latest_data = (
    df
    .sort_values("date")
    .groupby(["store", "item"])
    .tail(1)
)

latest_data.shape

(500, 20)

In [8]:
features = [
    "store",
    "item",

    "year",
    "month",
    "quarter",
    "day",
    "day_of_week",
    "day_of_year",
    "week_of_year",
    "is_weekend",

    "lag_7",
    "lag_14",
    "lag_30",
    "lag_90",

    "rolling_mean_7",
    "rolling_mean_30",

    "rolling_std_7",
    "rolling_std_30"
]

In [9]:
latest_data["forecast_demand"] = model.predict(
    latest_data[features]
)

In [10]:
latest_data[
    [
        "store",
        "item",
        "forecast_demand"
    ]
].head()

,store,item,forecast_demand
433999,5,50,49.410816
864527,10,48,56.520325
272551,4,7,58.499863
274287,4,8,78.635551
276023,4,9,52.392513


In [11]:
latest_data["avg_daily_demand"] = (
    latest_data["rolling_mean_30"]
)

In [12]:
latest_data["demand_std"] = (
    latest_data["rolling_std_30"]
)

In [13]:
LEAD_TIME = 7
SERVICE_LEVEL = 1.65

In [15]:
latest_data["safety_stock"] = (
    SERVICE_LEVEL * latest_data["demand_std"] * np.sqrt(LEAD_TIME)
)

In [16]:
latest_data["reorder_point"] = (
    latest_data["avg_daily_demand"]
    * LEAD_TIME
    + latest_data["safety_stock"]
)

In [17]:
np.random.seed(42)

latest_data["current_inventory"] = np.random.randint(
    50,
    300,
    size=len(latest_data)
)

In [18]:
def inventory_status(row):

    if row["current_inventory"] < row["reorder_point"]:
        return "Reorder Required"

    elif row["current_inventory"] > row["reorder_point"] * 2:
        return "Overstock"

    else:
        return "Healthy"

In [19]:
latest_data["inventory_status"] = (
    latest_data.apply(
        inventory_status,
        axis=1
    )
)

In [21]:
latest_data[
    [
        "store",
        "item",
        "forecast_demand",
        "safety_stock",
        "reorder_point",
        "current_inventory",
        "inventory_status"
    ]
].head(20)

,store,item,forecast_demand,safety_stock,reorder_point,current_inventory,inventory_status
433999,5,50,49.410816,37.996684,338.763351,152,Reorder Required
864527,10,48,56.520325,44.277927,388.911261,229,Reorder Required
272551,4,7,58.499863,38.293004,403.926337,142,Reorder Required
274287,4,8,78.635551,53.988794,547.255461,64,Reorder Required
276023,4,9,52.392513,42.901751,370.501751,156,Reorder Required
277759,4,10,75.492165,49.062587,520.629254,121,Reorder Required
279495,4,11,74.446983,45.695609,511.428942,238,Reorder Required
281231,4,12,74.340622,54.505058,510.205058,70,Reorder Required
282967,4,13,85.707886,58.043221,594.709888,152,Reorder Required
284703,4,14,60.197102,45.655324,417.121990,171,Reorder Required


In [22]:
latest_data["inventory_status"].value_counts()

inventory_status
Reorder Required    413
Healthy              84
Overstock             3
Name: count, dtype: int64

In [25]:
latest_data.to_csv(
    "../processed_data/inventory_recommendations.csv",
    index=False
)

print("Inventory Recommendations is saved succesfully")

Inventory Recommendations is saved succesfully
